# BOJ Swap Analysis: Trend-Following Strategy
予測値のトレンドに基づいた順張り戦略のバックテスト。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os, importlib
from catboost import CatBoostRegressor

if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')
sys.path.append(os.getcwd())
from src.processing import load_and_clean_data
from src.pooling import pool_boj_data

sns.set_theme(style='whitegrid')
plt.rcParams['font.family'] = 'Hiragino Sans'

In [ ]:
MAX_N = 5
df_cleaned = load_and_clean_data('data/BOJ_data.xlsx', 'data/BOJ_meeting_history.csv')
mpm_dates = pd.to_datetime(pd.read_csv('data/BOJ_meeting_history.csv')['Date'])
df_pooled = pool_boj_data(df_cleaned, mpm_dates=mpm_dates, max_n=MAX_N)

features = [c for c in df_pooled.columns if 'FracDiff' in c or 'Consec_Imp' in c or c in ['Days_to_MPM', 'Meeting_Index']]
train_final = df_pooled.dropna(subset=[f'Target_FD_{MAX_N}d'] + features)
split_date = sorted(train_final['Date'].unique())[int(len(train_final['Date'].unique()) * 0.8)]
test_df = df_pooled[df_pooled['Date'] >= split_date].copy()

for n in [1, 3, 5]:
    m = CatBoostRegressor(iterations=800, learning_rate=0.05, verbose=0, random_seed=42)
    m.fit(train_final[train_final['Date'] < split_date][features], train_final[train_final['Date'] < split_date][f'Target_FD_{n}d'])
    test_df[f'Pred_Rate_{n}d'] = m.predict(test_df[features]) - test_df[f'Mem_{n}d'].ffill()

In [ ]:
def plot_analysis(df, title):
    df = df.copy().sort_values('Date')
    
    # 1. シグナル判定 (Trend-Following)
    # 昨日予測した今日の値
    df['Pred_1d_lag1'] = df['Pred_Rate_1d'].shift(1)
    
    # 金利上昇にベッド (Short/Pay): 昨日(今日) < 今日(1d) < 今日(3d) < 今日(5d)
    cond_short = (df['Pred_1d_lag1'] < df['Pred_Rate_1d']) & \
                 (df['Pred_Rate_1d'] < df['Pred_Rate_3d']) & \
                 (df['Pred_Rate_3d'] < df['Pred_Rate_5d'])
    
    # 金利低下にベッド (Long/Recv): 昨日(今日) > 今日(1d) > 今日(3d) > 今日(5d)
    cond_long = (df['Pred_1d_lag1'] > df['Pred_Rate_1d']) & \
                (df['Pred_Rate_1d'] > df['Pred_Rate_3d']) & \
                (df['Pred_Rate_3d'] > df['Pred_Rate_5d'])
    
    df['Signal'] = 0
    df.loc[cond_long, 'Signal'] = 1   # Long (Recv)
    df.loc[cond_short, 'Signal'] = -1  # Short (Pay)
    
    # 2. 損益計算 (5日後にアンワインド)
    # Long(+1)の場合: 利益 = Swap_Rate(エントリー) - Actual_5d(クローズ)
    # Short(-1)の場合: 利益 = Actual_5d(クローズ) - Swap_Rate(エントリー)
    # まとめて: Signal * (Swap_Rate - Actual_5d)
    df['PnL'] = df['Signal'] * (df['Swap_Rate'] - df['Actual_5d'])
    
    # 3. 未来拡張 (可視化用)
    latest_t = df[df['Swap_Rate'].notnull()]['Date'].max()
    ext_dates = pd.bdate_range(start=latest_t + pd.Timedelta(days=1), periods=5)
    df_ext = pd.concat([df, pd.DataFrame({'Date': ext_dates})], ignore_index=True).sort_values('Date')
    
    # 描画
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 12), gridspec_kw={'height_ratios': [2, 1]}, sharex=True)
    
    # 上段: 価格推移
    ax1.plot(df_ext['Date'], df_ext['Swap_Rate'], color='black', linewidth=3, label='Actual')
    colors = sns.color_palette("Reds", 3)
    for i, n in enumerate([1, 3, 5]):
        ax1.plot(df_ext['Date'], df_ext[f'Pred_Rate_{n}d'].shift(n), label=f'Pred {n}d ago', linestyle='--', color=colors[i], alpha=0.7)
    
    longs = df_ext[df_ext['Signal'] == 1]; shorts = df_ext[df_ext['Signal'] == -1]
    ax1.scatter(longs['Date'], longs['Swap_Rate'], marker='^', color='blue', s=100, label='Long(Recv)', zorder=5)
    ax1.scatter(shorts['Date'], shorts['Swap_Rate'], marker='v', color='red', s=100, label='Short(Pay)', zorder=5)
    ax1.set_title(f'{title} Trajectory & Trend Signals', fontsize=16); ax1.legend(bbox_to_anchor=(1.05, 1))
    
    # 下段: 累積損益 (Cumulative PnL)
    ax2.plot(df_ext['Date'], df_ext['PnL'].fillna(0).cumsum(), color='green', linewidth=2)
    ax2.fill_between(df_ext['Date'], 0, df_ext['PnL'].fillna(0).cumsum(), color='green', alpha=0.1)
    ax2.set_title(f'{title} Cumulative PnL (5d Unwind)', fontsize=13)
    
    plt.tight_layout(); plt.show()
    return df_ext

# 各会合の結果表示
m1_res = plot_analysis(test_df[test_df['Meeting_Index'] == 1], 'M1')
m5_res = plot_analysis(test_df[test_df['Meeting_Index'] == 5], 'M5')